In [2]:
# it search for "android" in repository metadata, readme, and documentation files
# It uses multiple GitHub tokens to avoid rate limits
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token #{token_index + 1}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"

# === Load previous output if exists ===
if os.path.exists(output_path):
    print("🟡 Resuming from previous output...")
    df = pd.read_csv(output_path)
else:
    df = pd.read_csv(input_path)
    df["android_metadata_match"] = "none"
    df["android_in_readme"] = "none"
    df["Valid_Repo_Step3"] = df["Valid_Repo_Step2"]

# === Filter unprocessed repos ===
valid_df = df[(df["Valid_Repo_Step2"] == "yes") & (df["android_metadata_match"] == "none")].copy()
print(f"🔍 Reviewing {len(valid_df)} remaining repos...\n")

# === Helper: Search Android in readme or docs ===
def search_android_in_files(repo):
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(readme_url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
            if "android" in decoded:
                return True
        except:
            pass

    contents_url = f"https://api.github.com/repos/{repo}/contents"
    r = requests.get(contents_url, headers=get_headers())
    if r.status_code == 200:
        for item in r.json():
            name = item.get("name", "").lower()
            if name.endswith((".md", ".rst", ".markdown")):
                file_url = item.get("download_url")
                if file_url:
                    try:
                        text = requests.get(file_url, headers=get_headers()).text.lower()
                        if "android" in text:
                            return True
                    except:
                        pass

    wiki_url = f"https://raw.githubusercontent.com/wiki/{repo}/Home.md"
    r = requests.get(wiki_url, headers={"User-Agent": "android-repo-crawler/1.0"})
    if r.status_code == 200 and "android" in r.text.lower():
        return True

    return False

# === Process repos ===
for idx, row in enumerate(valid_df.itertuples(), start=1):
    repo = row.full_name
    print(f"🔎 [{idx}/{len(valid_df)}] Checking: {repo}")

    name = str(row.name).lower()
    topics = str(row.topics).lower()
    desc = str(getattr(row, "description", "")).lower()

    metadata_hit = "android" in name or "android" in topics or "android" in desc
    df.loc[df["full_name"] == repo, "android_metadata_match"] = "yes" if metadata_hit else "no"

    readme_hit = search_android_in_files(repo)
    df.loc[df["full_name"] == repo, "android_in_readme"] = "yes" if readme_hit else "no"

    # 💾 Save progress every 5 repos
    if idx % 5 == 0:
        df.to_csv(output_path, index=False)
        print("💾 Progress saved.")

# === Final Save ===
df.to_csv(output_path, index=False)
print(f"\n✅ Step 3 complete. Output saved to: {output_path}")


🟡 Resuming from previous output...
🔍 Reviewing 4863 remaining repos...

🔎 [1/4863] Checking: X-Wei/flutter_catalog
🔁 Using token #1


C:\Users\gilla\AppData\Local\Temp\ipykernel_20768\829571908.py:36: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(output_path)


🔁 Using token #2
🔁 Using token #3
🔁 Using token #4
🔎 [2/4863] Checking: Sub6Resources/flutter_html
🔁 Using token #5
🔁 Using token #6
🔁 Using token #1
🔁 Using token #2
🔁 Using token #3
🔎 [3/4863] Checking: duytq94/flutter-chat-demo
🔁 Using token #4
🔁 Using token #5
🔁 Using token #6
🔎 [4/4863] Checking: aagarwal1012/Animated-Text-Kit
🔁 Using token #1
🔁 Using token #2
🔁 Using token #3
🔁 Using token #4
🔁 Using token #5
🔁 Using token #6
🔁 Using token #1
🔎 [5/4863] Checking: flutter/devtools
🔁 Using token #2
🔁 Using token #3
🔁 Using token #4
🔁 Using token #5
🔁 Using token #6
💾 Progress saved.
🔎 [6/4863] Checking: aloisdeniel/figma-to-flutter
🔁 Using token #1
🔁 Using token #2
🔎 [7/4863] Checking: boeledi/Streams-Block-Reactive-Programming-in-Flutter
🔁 Using token #3
🔁 Using token #4
🔁 Using token #5
🔎 [8/4863] Checking: yako-dev/flutter_badges
🔁 Using token #6
🔁 Using token #1
🔁 Using token #2
🔎 [9/4863] Checking: Pyozer/introduction_screen
🔁 Using token #3
🔎 [10/4863] Checking: codegrue/card